# Truncated-model sensitivity under weak overlap

This notebook compares exact UKL and BKL fits with their truncated (bounded)
variants under the weak-overlap coverage-diagnostic design. The bounds are part
of the fitted model: the link saturates at the stated representer bounds, so no
fitted value is rewritten afterwards. Where a bound binds the estimator targets
a modified (bounded) estimand, so the bounded rows are a target-sensitivity
analysis around the exact rows, and the reported per-side binding rates state
how often each bound is active. Numerical failures remain in the denominator
through the stable count.

The propensity window is symmetric: `e_min` of 0.01, 0.02, and 0.05 corresponds
to representer magnitude caps of 100, 50, and 20. The true effect is constant
(one), so bias and coverage are measured against the population effect.

In [ ]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
for candidate in (REPO_ROOT, *REPO_ROOT.parents):
    if (candidate / "src" / "genriesz").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from inside the genriesz repository.")

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from genriesz import BoundedBKLGenerator, BoundedUKLGenerator
from genriesz.experiments import (
    CoverageDiagnosticBasis,
    branch_treated,
    fit_one_grr_with_basis,
    make_compatible_generator,
    make_coverage_diagnostic_data,
)

TABLE_CONFIG = {"float_format": "{:.4f}", "max_rows": 200}
PLOT_CONFIG = {
    "figure_size": (9.0, 5.2),
    "title_fontsize": 14,
    "axis_fontsize": 12,
    "tick_fontsize": 10,
    "legend_fontsize": 10,
    "grid_alpha": 0.30,
    "dpi": 140,
}
METHOD_COLORS = {"UKL": "#F58518", "BKL": "#54A24B"}
DISPLAY_LABELS = {"arw": "ARW"}


def label_of(value):
    return DISPLAY_LABELS.get(str(value), str(value))


def display_table(df: pd.DataFrame, *, caption: str | None = None, digits: int = 4):
    table = df.copy()
    for column in ("estimator", "loss"):
        if column in table.columns:
            table[column] = table[column].map(label_of)
    numeric = table.select_dtypes(include=[np.number]).columns
    table[numeric] = table[numeric].round(digits)
    styler = table.style.format(precision=digits)
    if caption is not None:
        styler = styler.set_caption(caption)
    display(styler)


pd.options.display.max_rows = TABLE_CONFIG["max_rows"]

In [ ]:
N_REPLICATIONS = 400
SAMPLE_SIZE = 2000
OVERLAP_SCALE = 2.5
RIEZ_LAMBDA = 1e-2
FOLDS = 5
LOSSES = ("UKL", "BKL")
BOUND_LEVELS = (("none", None), ("0.01", 0.01), ("0.02", 0.02), ("0.05", 0.05))
BOUND_ORDER = [label for label, _ in BOUND_LEVELS]

In [ ]:
def make_sensitivity_generator(loss: str, e_min: float | None):
    if e_min is None:
        return make_compatible_generator(loss, estimand="ATE")
    if loss == "UKL":
        return BoundedUKLGenerator.from_propensity_bounds(
            e_min, 1.0 - e_min, branch_fn=branch_treated
        )
    return BoundedBKLGenerator(C=1.0, alpha_max=1.0 / e_min, branch_fn=branch_treated)


sensitivity_rows = []
for replication in range(N_REPLICATIONS):
    data = make_coverage_diagnostic_data(
        n=SAMPLE_SIZE,
        seed=3_000_000 + 12_007 * replication,
        overlap_scale=OVERLAP_SCALE,
    )
    for loss in LOSSES:
        for bound_label, e_min in BOUND_LEVELS:
            fit_rows = fit_one_grr_with_basis(
                data,
                estimand="ATE",
                loss_spec={"label": loss, "loss": loss},
                representer_basis=CoverageDiagnosticBasis(include_quadratic=True),
                cross_fit=True,
                lam=RIEZ_LAMBDA,
                folds=FOLDS,
                estimators=("arw",),
                random_state=replication,
                label_info={"bound": bound_label, "replication": replication},
                generator_override=make_sensitivity_generator(loss, e_min),
            )
            sensitivity_rows.extend(fit_rows)
sensitivity_results = pd.DataFrame(sensitivity_rows)
sensitivity_results["bound"] = pd.Categorical(
    sensitivity_results["bound"], categories=BOUND_ORDER, ordered=True
)

In [ ]:
summary_rows = []
for keys, group in sensitivity_results.groupby(["loss", "bound"], dropna=False, observed=True):
    loss, bound = keys
    stable = group[group["status"] == "ok"].copy()
    row = {
        "loss": loss,
        "bound": bound,
        "stable": int(stable.shape[0]),
        "failures": int(group.shape[0] - stable.shape[0]),
    }
    if not stable.empty:
        row.update(
            {
                "bias": float(stable["error"].mean()),
                "sd": float(stable["estimate"].std(ddof=1)),
                "mean_se": float(stable["se"].mean()),
                "coverage": float(stable["covered"].mean()),
                "rmse": float(np.sqrt(stable["squared_error"].mean())),
                "alpha_abs_max": float(stable["alpha_abs_max"].mean()),
                "binding_lower": float(stable["riesz_binding_rate_lower_max"].mean()),
                "binding_upper": float(stable["riesz_binding_rate_upper_max"].mean()),
            }
        )
    summary_rows.append(row)
sensitivity_summary = pd.DataFrame(summary_rows).sort_values(["loss", "bound"])
display_table(
    sensitivity_summary,
    caption="Truncated-model sensitivity under weak overlap (ATE, ARW)",
)

In [ ]:
ok_rows = sensitivity_summary.dropna(subset=["coverage"])
figure, axis = plt.subplots(figsize=PLOT_CONFIG["figure_size"], dpi=PLOT_CONFIG["dpi"])
for loss in LOSSES:
    panel = ok_rows[ok_rows["loss"] == loss]
    axis.plot(
        panel["bound"].astype(str),
        panel["coverage"],
        marker="o",
        label=loss,
        color=METHOD_COLORS.get(loss),
    )
axis.axhline(0.95, color="black", linestyle="--", linewidth=1.0)
axis.set_xlabel("Propensity window e_min", fontsize=PLOT_CONFIG["axis_fontsize"])
axis.set_ylabel("Coverage of the population effect", fontsize=PLOT_CONFIG["axis_fontsize"])
axis.set_title(
    "Coverage across representer bounds (weak overlap)",
    fontsize=PLOT_CONFIG["title_fontsize"],
)
axis.legend(fontsize=PLOT_CONFIG["legend_fontsize"])
axis.grid(alpha=PLOT_CONFIG["grid_alpha"])
figure.tight_layout()
plt.show()